<a href="https://colab.research.google.com/github/Tioplate/AE-3-Summer-Short-Internship-2025-Team6/blob/main/Report/IMISToolA2026_Report4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Report 4 (2026/07/27 ver.A)

for Tools for intelligent interaction systems a (0ALE005 / 0AL5707).

---

* Student ID: (E.g. 202099998)...Erase-and-put-yours-here
* Name: (E.g. 筑波 太郎）...Erase-and-put-yours-here
* Colab account: (E.g. tsukuba.taro.xs@alumni.tsukuba.ac.jp)...Erase-and-put-yours-here

---

Report could be written in English or Japanese. / レポートの記述は日本語でも英語でもよい．

---

---
# Report4A: Live capture query with CIFAR-10 or MNIST  

* Show the way to build a classfier of CIFAR10 or MNIST.
* Provide a python program that captures an image from a camera and that shows its recognition result of the image immediately.

Hint: Each shown in tutorials. Find a way to unite them.



## 1. Building an MNIST Classifier

MNISTデータセットを用いて手書き数字0～9を分類するCNNを構築した。
以下のコードでは、学習データを用いてモデルを学習し、テストデータに対する正解率を確認する。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x


def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] "
                  f"Loss: {loss.item():.6f}")


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f"\nTest set: Average loss: {test_loss:.4f}, "
          f"Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n")


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    epochs = 3
    batch_size = 64
    test_batch_size = 1000
    lr = 1.0

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

    model = Net().to(device)
    optimizer = optim.Adadelta(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.7)

    for epoch in range(1, epochs + 1):
        train(model, device, train_loader, optimizer, epoch)
        test(model, device, test_loader)
        scheduler.step()

    torch.save(model.state_dict(), "mnist_cnn.pt")
    print("Model saved to mnist_cnn.pt")


if __name__ == "__main__":
    main()

## 2. Recognition of a Photographed Handwritten Digit

カメラで撮影した白紙上の手書き数字画像をファイルとして読み込み、MNISTの入力形式に近づけるための前処理を行った。
グレースケール化、二値化、白黒反転、数字部分の抽出、サイズ変更、中央配置を行い、28×28画素の画像として学習済みモデルに入力した。

In [ ]:
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x


def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def take_photo_local(filename='photo.jpg'):
    try:
        import cv2
    except ImportError:
        raise RuntimeError(
            "Local capture requires opencv-python. Please run: pip install opencv-python"
        )

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Cannot open camera. Check if it's in use or permission is denied.")

    print("Camera window opened: press SPACE to capture, Q/ESC to quit.")
    saved = False
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                raise RuntimeError("Failed to read frame from camera.")
            cv2.imshow('Press SPACE to capture, Q/ESC to quit', frame)
            key = cv2.waitKey(1) & 0xFF
            if key == 32:
                cv2.imwrite(filename, frame)
                saved = True
                break
            elif key in (27, ord('q')):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

    if not saved:
        raise RuntimeError("No photo captured, cancelled by user.")
    return filename


def take_photo(filename='photo.jpg', quality=0.8):
    if not _is_colab():
        return take_photo_local(filename)

    from IPython.display import display, Javascript
    from google.colab.output import eval_js
    from base64 import b64decode

    js = Javascript('''
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const capture = document.createElement('button');
          capture.textContent = 'Capture';
          div.appendChild(capture);

          const video = document.createElement('video');
          video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});

          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();

          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

          await new Promise((resolve) => capture.onclick = resolve);

          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }
        ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename


def _otsu_threshold(gray_arr):
    hist, _ = np.histogram(gray_arr.flatten(), bins=256, range=(0, 256))
    total = gray_arr.size
    sum_total = np.dot(np.arange(256), hist)
    sum_b, w_b, max_var, thresh_val = 0.0, 0.0, 0.0, 0
    for t in range(256):
        w_b += hist[t]
        if w_b == 0:
            continue
        w_f = total - w_b
        if w_f == 0:
            break
        sum_b += t * hist[t]
        m_b = sum_b / w_b
        m_f = (sum_total - sum_b) / w_f
        var_between = w_b * w_f * (m_b - m_f) ** 2
        if var_between > max_var:
            max_var = var_between
            thresh_val = t
    return thresh_val


def _crop_center_roi(gray_arr, ratio=0.7):
    h, w = gray_arr.shape
    new_h, new_w = int(h * ratio), int(w * ratio)
    y_off = (h - new_h) // 2
    x_off = (w - new_w) // 2
    roi = gray_arr[y_off:y_off + new_h, x_off:x_off + new_w]
    return roi, x_off, y_off


def _get_digit_mask(gray_arr):
    import cv2

    blurred = cv2.GaussianBlur(gray_arr, (5, 5), 0)
    block_size = max(11, (min(gray_arr.shape) // 8) | 1)
    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, block_size, 5
    )
    kernel = np.ones((3, 3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    return binary > 0


def _keep_largest_component(digit_mask):
    import cv2

    mask_uint8 = digit_mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_uint8, connectivity=8)

    if num_labels <= 1:
        return digit_mask

    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_label = 1 + int(np.argmax(areas))
    clean_mask = labels == largest_label
    return clean_mask


def preprocess_image(filename='photo.jpg', show_steps=True):
    from PIL import ImageFilter

    img = Image.open(filename).convert('RGB')

    gray = img.convert('L')
    gray_arr = np.array(gray)

    roi_arr, x_off, y_off = _crop_center_roi(gray_arr, ratio=0.7)

    digit_mask = _get_digit_mask(roi_arr)
    digit_mask = _keep_largest_component(digit_mask)

    ys, xs = np.where(digit_mask)
    if len(xs) == 0 or len(ys) == 0:
        x0, y0 = 0, 0
        x1, y1 = roi_arr.shape[1] - 1, roi_arr.shape[0] - 1
    else:
        x0, x1 = int(xs.min()), int(xs.max())
        y0, y1 = int(ys.min()), int(ys.max())

    binary_arr = (digit_mask * 255).astype(np.uint8)

    import cv2
    dilate_kernel = np.ones((3, 3), np.uint8)
    binary_arr = cv2.dilate(binary_arr, dilate_kernel, iterations=1)

    binary_img = Image.fromarray(binary_arr)
    cropped = binary_img.crop((x0, y0, x1 + 1, y1 + 1))

    w, h = cropped.size
    scale = 20.0 / max(w, h, 1)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized_digit = cropped.resize((new_w, new_h), Image.LANCZOS)

    digit_arr = np.array(resized_digit).astype(np.float64)
    total_mass = digit_arr.sum()
    if total_mass > 0:
        ys_idx, xs_idx = np.indices(digit_arr.shape)
        cy = (ys_idx * digit_arr).sum() / total_mass
        cx = (xs_idx * digit_arr).sum() / total_mass
    else:
        cy, cx = new_h / 2.0, new_w / 2.0

    canvas = Image.new('L', (28, 28), color=0)
    paste_x = int(round(14 - cx))
    paste_y = int(round(14 - cy))
    canvas.paste(resized_digit, (paste_x, paste_y))

    final_img = canvas.filter(ImageFilter.GaussianBlur(radius=1))

    final_arr = np.array(final_img).astype(np.float32)
    max_val = final_arr.max()
    if max_val > 0:
        final_arr = final_arr * (255.0 / max_val)
        final_arr = np.clip(final_arr, 0, 255).astype(np.uint8)
        final_img = Image.fromarray(final_arr)

    if show_steps:
        fig, axes = plt.subplots(1, 4, figsize=(12, 3))
        axes[0].imshow(img)
        axes[1].imshow(roi_arr, cmap='gray')
        axes[2].imshow(cropped, cmap='gray')
        axes[3].imshow(final_img, cmap='gray')
        for ax in axes:
            ax.axis('off')
        plt.show()

    arr = np.array(final_img).astype(np.float32) / 255.0
    arr = (arr - 0.1307) / 0.3081
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
    return tensor


def load_model(weights_path='mnist_cnn.pt', device='cpu'):
    model = Net().to(device)
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.eval()
    return model


def capture_and_predict(weights_path='mnist_cnn.pt', filename='photo.jpg'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("Please click the Capture button in the popup...")
    take_photo(filename)

    print("Processing photo...")
    x = preprocess_image(filename).to(device)

    model = load_model(weights_path, device)

    with torch.no_grad():
        output = model(x)
        probs = F.softmax(output, dim=1)
        pred = output.argmax(dim=1).item()
        confidence = probs[0, pred].item()

    print(f"Prediction: {pred}  (confidence: {confidence:.2%})")
    return pred


if __name__ == '__main__':
    capture_and_predict()

## 目的

本課題では、MNISTデータセットを用いて手書き数字の分類器を構築し、カメラで撮影した手書き数字の画像に対して認識を行った。

## 方法

まず、MNISTの学習データを用いて数字0～9を分類するモデルを学習した。学習後、MNISTのテストデータを用いて分類性能を確認した。

次に、カメラで撮影した白紙上の手書き数字画像を入力として使用した。撮影画像はMNISTの画像形式と大きく異なるため、以下の前処理を行った。

* グレースケールへの変換
* 二値化および白黒反転
* 数字部分の抽出
* アスペクト比を維持した縮小
* 28×28画素の画像中央への配置

これにより、撮影した画像をMNISTの学習画像に近い形式へ変換し、学習済みモデルに入力して認識結果を表示した。

## 結果

MNISTのテストデータに対して、分類器は 99.11%の正解率を得た。
また、実際に撮影した手書き数字「6」の画像に対して前処理を行い、分類器に入力した結果、「6」（確信度85.58%） と正しく認識することができた。
撮影した元画像と比較すると、前処理後の画像は黒背景・白文字の28×28画素画像となり、MNISTの画像に近い形式になっていることを確認した。

## 考察

MNISTのテストデータでは高い認識精度が得られた一方で、最初は実際に撮影した画像に対する認識精度が低かった。これは、背景、明るさ、数字の大きさや位置などがMNISTの画像と異なっていたためであると考えられる。

そこで、数字部分の抽出、二値化、サイズ変更、中央配置などの前処理を行ったところ、認識結果を改善することができた。この実験から、学習済みモデルを実際の画像に適用する場合には、入力画像を学習データと近い形式に変換することが重要であると分かった。

特に、撮影画像の背景に生じる暗部（周辺減光）が誤って前景と判定されたことや、平滑化処理により線の輝度ピークが低下し、認識精度が低下した点が課題であった。これらは中心領域のクロップ、局所適応二値化、コントラスト正規化を導入することで解決した。

---
# Report4B: Tutorials of pytorch

There are [many tutorials](https://github.com/kameda-yoshinari/IMISToolExeA2021/blob/master/300_PyTorch.ipynb) provided by the Pytorch official site. Pick up **two tutorials** (you should complete those tutorials) and make a report for the two tutorials on:

* Summary (2-3 lines)
* What you learn
* The most difficult part

Note that you are encouraged to choose the recommended two .


## Tutorial 1: Deep Learning with PyTorch: A 60 Minute Blitz

### Summary

本チュートリアルでは、PyTorchにおけるTensorの基本操作、Autograd、ニューラルネットワークの構築方法、画像分類モデルの学習手順について学習した。最後にCIFAR-10データセットを用いてCNNによる画像分類器を構築し、学習から評価までの一連の流れを確認した。

### What I learned

Tensor、Autograd、`nn.Module`、損失関数、Optimizerが深層学習の学習過程でどのように関係しているかを理解した。特に、`zero_grad()` → forward → loss計算 → `backward()` → `step()`というPyTorchにおける基本的な学習処理の流れを学んだ。また、学習時と推論時で入力データに同じ前処理を適用することの重要性についても理解した。

### The most difficult part

最も難しかったのは、Autogradによる勾配計算とパラメータ更新の関係を理解することであった。最初は、`loss.backward()`によってなぜ各層のパラメータの勾配を自動的に計算できるのか、また各学習ステップの前に`optimizer.zero_grad()`が必要である理由が分かりにくかった。チュートリアルを通して、PyTorchが計算過程を記録して逆伝播を行うこと、および勾配がデフォルトでは累積されるため、更新前に勾配を初期化する必要があることを理解した。

## Tutorial 2: Learning PyTorch with Examples

### Summary

本チュートリアルでは、三次多項式による関数近似を共通の例として用い、NumPyによる実装からPyTorchのTensor、Autograd、`nn.Module`、Optimizerを利用した実装へ段階的に移行することで、PyTorchの基本機能について学習した。

### What I learned

同じ学習問題でも、NumPyでは順伝播、勾配計算、パラメータ更新を自分で実装する必要があるのに対し、PyTorchではAutogradによって勾配を自動計算できることを学んだ。また、`nn.Module`を利用することでモデルとパラメータを管理し、`torch.optim`を利用することでパラメータの更新処理も簡潔に記述できることを理解した。

### The most difficult part

最も難しかったのは、手動で勾配を計算する方法からAutogradによる自動微分へ移行する部分であった。特に、`loss.backward()`によって計算グラフを逆方向にたどり、各パラメータの勾配が自動的に求められるという仕組みを理解するのが難しかった。しかし、NumPyによる手動計算と比較することで、Autogradを利用する利点を理解することができた。

---
# Report4C: AI support to learn python and PyTorch  

Find out the state-of-the-art AI support which is available / helpful on learning how to write python / pytorch codes on google colab.
Then write your own comment on what kind of changes would happen in a couple of years.

* AI service name
* AI service "official web site"
* Major funcitons of provided support
* The service start date and version history (including the URL that holds such info)


## AI service name

GitHub Copilot

## Official web site

https://github.com/copilot

## Major functions of provided support

GitHub Copilotは、プログラミングを支援する生成AIサービスである。主な機能として、入力中のコードの補完、自然言語によるプログラミングに関する質問への回答、コードの説明、バグ修正の提案、テストコードの生成などがある。また、現在ではAgent modeなども提供されており、複数のファイルを確認しながらコードを編集したり、必要な処理を提案したりすることも可能である。

PythonはGitHub Copilotが特に得意とする言語の一つであり、PyTorchを利用したプログラムについても、モデルの定義、学習ループ、データ前処理などのコード生成や説明に利用できる。

Google Colab自体にはGitHub Copilotが直接統合されているわけではないが、GitHub Copilot Chatなどを利用してPythonやPyTorchのコードを生成・修正し、そのコードをGoogle Colab上で実行して確認するという利用方法が考えられる。このように、コードの作成だけでなく、既存コードの意味の理解やエラー原因の調査にも利用できるため、プログラミング学習の補助として有用である。

## Service start date and version history

GitHub Copilotは、2021年6月29日にTechnical Previewとして公開された。その後、2022年6月に一般提供（General Availability）が開始された。

初期のGitHub Copilotは主にIDE上でコードを自動補完する「AI pair programmer」として提供されていたが、その後Copilot Chat、コードレビュー、Agent mode、CLI、複数のAIモデルの選択など、多くの機能が追加されている。

現在も機能追加や改善が頻繁に行われており、更新履歴はGitHub ChangelogのCopilotカテゴリで確認することができる。

### References for version history

- GitHub Blog: Introducing GitHub Copilot: your AI pair programmer  
  https://github.blog/news-insights/product-news/introducing-github-copilot-ai-pair-programmer/
- GitHub Blog: GitHub Copilot is generally available to all developers  
  https://github.blog/news-insights/product-news/github-copilot-is-generally-available-to-all-developers/
- GitHub Changelog: Copilot  
  https://github.blog/changelog/label/copilot/

## My comment

今後数年間で、PythonやPyTorchの学習方法は、文法やAPIをすべて記憶してからプログラムを書く方法から、AIと対話しながら必要なコードを作成し、その意味を確認しながら学習する方法へ変化していくと考える。

特に初心者にとっては、エラーメッセージの意味を調べたり、ライブラリの使用方法を検索したりする時間を減らし、実際にコードを動かしながら学習できることが大きな利点である。また、PyTorchのようにライブラリの機能が多い場合でも、目的を自然言語で説明することで基本的なコード例を得ることができるため、学習開始時の負担は小さくなると考えられる。

一方で、AIが生成したコードが常に正しいとは限らない。そのため、将来的には単にAIにコードを書かせる能力よりも、生成されたコードの意味を理解し、実行結果を確認し、誤りを判断する能力がより重要になると考える。

---
# Report submission

As for Report4, in addition to submit this ipynb, you are also asked to submit 801_miniGPT.ipynb and 802_miniGPT.ipnyb (in total of three files).

You should save this file as a report templete to your local google colaboratory folder and then edit it to fit your report.

The report submission should be made at this cource (0ALE005) at https://manaba.tsukuba.ac.jp .  
Note that 0AL5707 is coupled with 0ALE005 on manaba system, so 0AL5707 students should also submit the report at 0ALE005.  
File extension should be **ipynb**. Other format won't be accepted.  







---
Tools and Practices for Intelligent Interaction Systems A  
Master's and Docotal programs in intelligent and mechanical interaction systems, University of Tsukuba, Japan.  
KAMEDA Yoshinari, SHIBUYA Takeshi  

知能システムツール演習a  
知能機能システム学位プログラム (筑波大学大学院)  
担当：亀田能成，澁谷長史  

2026/07/27. Ver.B. 説明簡略化 / Simplified Explanation  
2026/07/27. Ver.A.  


